In [1]:
"""
B4. Unitary U_qr and U_ur(LCU) circuit verification and sweep benchmarks
------------------------------------------------------------------------
This script tests and benchmarks quantum circuit constructions for the
unitary U_qr operator and the U_ur operator implemented through a linear
combination of unitaries (LCU) scheme.

Implementation note
-------------------
Both circuit builders are imported from quantum.unitary_ur. The loader
implementation is selected per sweep from within the ``if __name__ == "__main__"``
blocks by passing

    loader_mode="dense"

or

    loader_mode="mcx"

into the sweep/test helpers. This makes it easy to run different tests
with different loader realizations without changing the imported module.

Benchmark and verification workflow
-----------------------------------
The script generates representative nonuniform node/index data using
perturbed uniform nodes and nearest-grid assignments, constructs
analytic reference amplitudes from y_from_t_and_s, phi_from_y,
theta_from_y, and alpha_vec_prime, and compares those references
against statevector simulation results produced by U_qr_circuit_no_x
and U_ur_lcu_circuit.

For parameter sweeps over circuit and approximation settings
(n, m, p, q, r, K), it measures:

    - relative ℓ2 state-amplitude error,
    - fidelity after global-phase alignment,
    - circuit build, simulation, verification, and plotting runtimes,
    - circuit resource statistics such as qubit count, depth, and size.

The script also supports shared caching of generated data and analytic
quantities, prints formatted console summaries and worst-case tables,
and optionally saves publication-style amplitude comparison figures and
tabulated sweep outputs.
"""

import _init_path

import itertools
import time
from contextlib import contextmanager
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector

from quantum.unitary_ur import (
    U_qr_circuit_no_x,
    U_ur_lcu_circuit,
    alpha_vec_prime,
    phi_from_y,
    theta_from_y,
    y_from_t_and_s,
)

from datasets.data import (
    nearest_grid_indices,
    nodes_perturbed_uniform,
    nodes_stratified_residuals,
)
from utils import _apply_publication_rcparams, _force_opaque


# =============================================================================
# Paths
# =============================================================================
FIG_DIR = Path("figures") / "B4_unitary_ur"


def _ensure_dirs() -> None:
    """Ensure required figure output directories exist.

    This routine verifies that the root figures directory is present and
    creates the benchmark-specific subdirectory if needed.

    Args:
        None.

    Returns:
        None.

    Raises:
        FileNotFoundError: If the root "figures/" directory does not exist.
    """
    root = Path("figures")
    if not root.exists():
        raise FileNotFoundError("Create 'figures/' directory first (e.g. `mkdir figures`).")
    FIG_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# Timing utility
# =============================================================================
@contextmanager
def timer():
    """Provide a simple elapsed-time callback within a context manager.

    This routine records a start timestamp and yields a callable that
    returns the elapsed wall-clock time when invoked.

    Args:
        None.

    Returns:
        A context manager yielding a zero-argument callable that returns
        elapsed time in seconds.

    Raises:
        None.
    """
    t0 = time.perf_counter()
    yield lambda: time.perf_counter() - t0


# =============================================================================
# Console / reporting helpers
# =============================================================================
def print_rule(char: str = "=", n: int = 88) -> None:
    """Print a repeated-character horizontal rule.

    This routine prints a single line consisting of the specified
    character repeated a fixed number of times.

    Args:
        char: Character used to build the rule.
        n: Number of repetitions.

    Returns:
        None.

    Raises:
        None.
    """
    print(char * n)


def print_section(title: str) -> None:
    """Print a section heading with surrounding rules.

    This routine prints a blank line, a horizontal rule, the section
    title, and a closing horizontal rule.

    Args:
        title: Section title to display.

    Returns:
        None.

    Raises:
        None.
    """
    print()
    print_rule("=")
    print(title)
    print_rule("=")


def print_subsection(title: str) -> None:
    """Print a subsection heading with surrounding rules.

    This routine prints a blank line, a lighter horizontal rule, the
    subsection title, and a closing rule.

    Args:
        title: Subsection title to display.

    Returns:
        None.

    Raises:
        None.
    """
    print()
    print_rule("-")
    print(title)
    print_rule("-")


def _fmt_float(x: Any, ndigits: int = 10) -> str:
    """Format a value as a fixed-point floating-point string.

    This routine converts the input to float when possible and formats it
    with a specified number of decimal places. Unsupported values are
    converted with str, and None is rendered as an empty string.

    Args:
        x: Value to format.
        ndigits: Number of digits after the decimal point.

    Returns:
        A formatted string representation of the value.

    Raises:
        ValueError: If ndigits is negative.
    """
    if ndigits < 0:
        raise ValueError("ndigits must be >= 0.")
    if x is None:
        return ""
    try:
        return f"{float(x): .{ndigits}f}"
    except Exception:
        return str(x)


def _fmt_sci(x: Any, ndigits: int = 3) -> str:
    """Format a value in scientific notation.

    This routine converts the input to float when possible and formats it
    in scientific notation. Unsupported values are converted with str,
    and None is rendered as an empty string.

    Args:
        x: Value to format.
        ndigits: Number of digits after the decimal point.

    Returns:
        A formatted scientific-notation string.

    Raises:
        None.
    """
    if x is None:
        return ""
    try:
        return f"{float(x):.{ndigits}e}"
    except Exception:
        return str(x)


def _fmt_int(x: Any) -> str:
    """Format a value as an integer string.

    This routine converts the input to int when possible. Unsupported
    values are converted with str, and None is rendered as an empty
    string.

    Args:
        x: Value to format.

    Returns:
        A string representation of the integer value.

    Raises:
        None.
    """
    if x is None:
        return ""
    try:
        return str(int(x))
    except Exception:
        return str(x)


def _print_ascii_table(headers: List[str], aligns: List[str], rows: List[List[str]]) -> None:
    """Print a Unicode ASCII-style table to the console.

    This routine computes column widths, applies per-column alignment,
    and renders a boxed table using Unicode line-drawing characters.

    Args:
        headers: Column header labels.
        aligns: Alignment specifiers for each column, typically "L" or "R".
        rows: Table body rows as lists of strings.

    Returns:
        None.

    Raises:
        None.
    """
    widths = [len(h) for h in headers]
    for row in rows:
        for j, cell in enumerate(row):
            widths[j] = max(widths[j], len(cell))

    TL, TR, BL, BR = "┌", "┐", "└", "┘"
    H, V = "─", "│"
    TJ, BJ, LJ, RJ, CJ = "┬", "┴", "├", "┤", "┼"

    def hline(left: str, mid: str, right: str) -> str:
        return left + mid.join(H * (w + 2) for w in widths) + right

    def format_cell(text: str, w: int, align: str, header: bool = False) -> str:
        if header:
            return f" {text:^{w}} "
        if align == "R":
            return f" {text:>{w}} "
        return f" {text:<{w}} "

    def render_row(row: List[str], header: bool = False) -> str:
        return V + V.join(
            format_cell(row[j], widths[j], aligns[j], header=header)
            for j in range(len(headers))
        ) + V

    print(hline(TL, TJ, TR))
    print(render_row(headers, header=True))
    print(hline(LJ, CJ, RJ))
    for row in rows:
        print(render_row(row))
    print(hline(BL, BJ, BR))


def extract_worst_cases(df: pd.DataFrame, metric: str, top_k: int = 20) -> pd.DataFrame:
    """Extract the worst-performing rows according to a metric.

    This routine removes rows with missing metric values, sorts the
    remaining rows in descending order by the requested metric, and
    returns the top-k entries.

    Args:
        df: Input results DataFrame.
        metric: Column name used for ranking.
        top_k: Number of worst cases to return.

    Returns:
        A DataFrame containing the top-k worst cases.

    Raises:
        None.
    """
    df_ok = df.dropna(subset=[metric]).copy()
    return df_ok.sort_values(metric, ascending=False).head(top_k).reset_index(drop=True)


def print_worst_cases_table_uqr(worst: "pd.DataFrame | List[dict]") -> None:
    """Print a formatted worst-case table for U_qr sweep results.

    This routine accepts either a pandas DataFrame or a list of
    dictionaries and prints key U_qr error and resource statistics in a
    fixed console table format.

    Args:
        worst: Worst-case rows as a DataFrame or list of dictionaries.

    Returns:
        None.

    Raises:
        ValueError: If worst is neither a DataFrame nor a list of dictionaries.
    """
    if isinstance(worst, pd.DataFrame):
        rows = worst.to_dict(orient="records")
    elif isinstance(worst, list) and all(isinstance(r, dict) for r in worst):
        rows = worst
    else:
        raise ValueError("worst must be a pandas DataFrame or a list of dictionaries.")

    headers = ["#", "n", "m", "p", "q", "l2_error", "fidelity", "qubits", "depth", "size"]
    aligns = ["R", "R", "R", "R", "R", "R", "R", "R", "R", "R"]

    table_rows: List[List[str]] = []
    for i, r_ in enumerate(rows, 1):
        table_rows.append(
            [
                str(i),
                _fmt_int(r_.get("n")),
                _fmt_int(r_.get("m")),
                _fmt_int(r_.get("p")),
                _fmt_int(r_.get("q")),
                _fmt_float(r_.get("l2_error"), 10),
                _fmt_float(r_.get("fidelity"), 10),
                _fmt_int(r_.get("qubits")),
                _fmt_int(r_.get("depth")),
                _fmt_int(r_.get("size")),
            ]
        )
    _print_ascii_table(headers, aligns, table_rows)


def print_worst_cases_table_uur(worst: "pd.DataFrame | List[dict]") -> None:
    """Print a formatted worst-case table for U_ur LCU sweep results.

    This routine accepts either a pandas DataFrame or a list of
    dictionaries and prints key U_ur LCU error and resource statistics in
    a fixed console table format.

    Args:
        worst: Worst-case rows as a DataFrame or list of dictionaries.

    Returns:
        None.

    Raises:
        ValueError: If worst is neither a DataFrame nor a list of dictionaries.
    """
    if isinstance(worst, pd.DataFrame):
        rows = worst.to_dict(orient="records")
    elif isinstance(worst, list) and all(isinstance(r, dict) for r in worst):
        rows = worst
    else:
        raise ValueError("worst must be a pandas DataFrame or a list of dictionaries.")

    headers = [
        "#", "n", "m", "p", "r", "K", "alpha_ur",
        "l2_error", "fidelity", "qubits", "depth", "size"
    ]
    aligns = ["R", "R", "R", "R", "R", "R", "R", "R", "R", "R", "R", "R"]

    table_rows: List[List[str]] = []
    for i, r_ in enumerate(rows, 1):
        table_rows.append(
            [
                str(i),
                _fmt_int(r_.get("n")),
                _fmt_int(r_.get("m")),
                _fmt_int(r_.get("p")),
                _fmt_int(r_.get("r")),
                _fmt_int(r_.get("K")),
                _fmt_float(r_.get("alpha_ur"), 10),
                _fmt_float(r_.get("l2_error"), 10),
                _fmt_float(r_.get("fidelity"), 10),
                _fmt_int(r_.get("qubits")),
                _fmt_int(r_.get("depth")),
                _fmt_int(r_.get("size")),
            ]
        )
    _print_ascii_table(headers, aligns, table_rows)


def print_sweep_summary(df: pd.DataFrame, label: str) -> None:
    """Print a compact summary table for a completed sweep.

    This routine computes aggregate error, fidelity, and runtime
    statistics for a sweep result DataFrame and prints them in a
    single-row summary table.

    Args:
        df: Sweep results DataFrame.
        label: Label identifying the sweep.

    Returns:
        None.

    Raises:
        None.
    """
    if df.empty:
        print(f"{label}: no rows.")
        return

    summary_rows = [[
        label,
        _fmt_int(len(df)),
        _fmt_sci(df["l2_error"].max(), 3),
        _fmt_sci(df["l2_error"].mean(), 3),
        _fmt_float(df["fidelity"].min(), 10),
        _fmt_float(df["fidelity"].mean(), 10),
        _fmt_float(df["build_s"].sum(), 3),
        _fmt_float(df["sim_s"].sum(), 3),
        _fmt_float(df["total_point_s"].sum(), 3),
    ]]
    headers = [
        "sweep", "cases", "max l2", "mean l2",
        "min fidelity", "mean fidelity",
        "Σ build [s]", "Σ sim [s]", "Σ total [s]"
    ]
    aligns = ["L", "R", "R", "R", "R", "R", "R", "R", "R"]
    _print_ascii_table(headers, aligns, summary_rows)


def make_display_df_uqr(df: pd.DataFrame) -> pd.DataFrame:
    """Build a display-oriented DataFrame for U_qr results.

    This routine selects the most relevant U_qr result columns and sorts
    the rows by sweep parameters to produce a stable display table.

    Args:
        df: Full U_qr results DataFrame.

    Returns:
        A reduced and sorted U_qr display DataFrame.

    Raises:
        None.
    """
    cols = [
        "n", "m", "p", "q",
        "l2_error", "fidelity",
        "build_s", "sim_s", "verify_s", "total_point_s",
        "qubits", "depth", "size",
    ]
    out = df[cols].copy()
    return out.sort_values(["n", "m", "p", "q"]).reset_index(drop=True)


def make_display_df_uur(df: pd.DataFrame) -> pd.DataFrame:
    """Build a display-oriented DataFrame for U_ur LCU results.

    This routine selects the most relevant U_ur LCU result columns and
    sorts the rows by sweep parameters to produce a stable display table.

    Args:
        df: Full U_ur LCU results DataFrame.

    Returns:
        A reduced and sorted U_ur LCU display DataFrame.

    Raises:
        None.
    """
    cols = [
        "n", "m", "p", "r", "K", "alpha_ur",
        "l2_error", "fidelity",
        "build_s", "sim_s", "verify_s", "total_point_s",
        "qubits", "depth", "size",
    ]
    out = df[cols].copy()
    return out.sort_values(["n", "m", "p", "r", "K"]).reset_index(drop=True)


# =============================================================================
# Analytic reference values
# =============================================================================
def u_qr_ideal_from_phi_theta(phi: np.ndarray, theta: np.ndarray, q: int) -> np.ndarray:
    """Compute ideal U_qr amplitudes from phase-angle data.

    This routine evaluates the analytic expression for U_qr using the
    supplied phi and theta angle arrays and a fixed order parameter q.

    Args:
        phi: Phase-angle array.
        theta: Theta-angle array.
        q: Order parameter used in the cosine factor.

    Returns:
        A complex-valued NumPy array of ideal U_qr amplitudes.

    Raises:
        None.
    """
    return np.exp(-1j * phi) * np.cos(float(q) * theta)


def u_qr_ideal(t: np.ndarray, s: np.ndarray, N: int, q: int) -> np.ndarray:
    """Compute ideal U_qr amplitudes from node and index data.

    This routine constructs y, phi, and theta from t and s, then
    evaluates the analytic U_qr reference vector.

    Args:
        t: Nonuniform node array.
        s: Nearest-grid index array.
        N: Problem size.
        q: Order parameter used in the cosine factor.

    Returns:
        A complex-valued NumPy array of ideal U_qr amplitudes.

    Raises:
        None.
    """
    y = y_from_t_and_s(t, s, N)
    phi = phi_from_y(y, N)
    theta = theta_from_y(y, N)
    return u_qr_ideal_from_phi_theta(phi, theta, q)


def u_r_ideal(t: np.ndarray, s: np.ndarray, N: int, r: int, K: int) -> np.ndarray:
    """Compute ideal U_r amplitudes using an LCU combination of U_qr terms.

    This routine evaluates ideal U_qr vectors for q = 0, ..., K - 1,
    combines them with alpha-prime coefficients, and returns the analytic
    U_r reference vector.

    Args:
        t: Nonuniform node array.
        s: Nearest-grid index array.
        N: Problem size.
        r: Target approximation order.
        K: Number of LCU terms.

    Returns:
        A complex-valued NumPy array of ideal U_r amplitudes.

    Raises:
        None.
    """
    uq = np.vstack([u_qr_ideal(t, s, N, q=q) for q in range(K)])
    coeffs = alpha_vec_prime(r=r, K=K)[:, None]
    return np.sum(coeffs * uq, axis=0)


# =============================================================================
# Shared cache
# =============================================================================
@dataclass
class SweepCache:
    """Cache shared t/s data and analytic reference quantities for sweeps.

    This data structure stores generated node/index pairs, intermediate
    angle data, ideal U_qr vectors, alpha-prime coefficients, and ideal
    U_r vectors so repeated sweep evaluations can reuse previously
    computed values.

    Args:
        None.

    Returns:
        A SweepCache instance with empty per-key caches initialized by
        default.

    Raises:
        None.
    """

    ts_by_n: Dict[int, Tuple[np.ndarray, np.ndarray]] = field(default_factory=dict)
    y_by_n: Dict[int, np.ndarray] = field(default_factory=dict)
    phi_by_n: Dict[int, np.ndarray] = field(default_factory=dict)
    theta_by_n: Dict[int, np.ndarray] = field(default_factory=dict)
    uqr_ideal_by_nq: Dict[Tuple[int, int], np.ndarray] = field(default_factory=dict)
    alpha_prime_by_rK: Dict[Tuple[int, int], np.ndarray] = field(default_factory=dict)
    ur_ideal_by_nrK: Dict[Tuple[int, int, int], np.ndarray] = field(default_factory=dict)

    def get_ts(
        self,
        n: int,
        make_t_s: Callable[[int, "TSConfig"], Tuple[np.ndarray, np.ndarray]],
        ts_cfg: "TSConfig",
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Retrieve cached or newly generated (t, s) data for a given n.

        This routine returns a cached pair of node and nearest-grid index
        arrays, generating and storing them on first access.

        Args:
            n: Log2 problem size.
            make_t_s: Callable that generates (t, s) from N and TSConfig.
            ts_cfg: Configuration used for data generation.

        Returns:
            A tuple (t, s) of NumPy arrays.

        Raises:
            None.
        """
        if n not in self.ts_by_n:
            self.ts_by_n[n] = make_t_s(2**n, ts_cfg)
        return self.ts_by_n[n]

    def ensure_angle_data(
        self,
        n: int,
        make_t_s: Callable[[int, "TSConfig"], Tuple[np.ndarray, np.ndarray]],
        ts_cfg: "TSConfig",
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """Ensure cached y, phi, and theta arrays exist for a given n.

        This routine retrieves the associated (t, s) arrays, computes
        y, phi, and theta if they are not already cached, and returns all
        related arrays.

        Args:
            n: Log2 problem size.
            make_t_s: Callable that generates (t, s) from N and TSConfig.
            ts_cfg: Configuration used for data generation.

        Returns:
            A tuple (t, s, y, phi, theta) of NumPy arrays.

        Raises:
            None.
        """
        t, s = self.get_ts(n, make_t_s, ts_cfg)
        N = 2**n

        if n not in self.y_by_n:
            y = y_from_t_and_s(t, s, N)
            self.y_by_n[n] = y
            self.phi_by_n[n] = phi_from_y(y, N)
            self.theta_by_n[n] = theta_from_y(y, N)

        return t, s, self.y_by_n[n], self.phi_by_n[n], self.theta_by_n[n]

    def get_uqr_ideal(
        self,
        n: int,
        q: int,
        make_t_s: Callable[[int, "TSConfig"], Tuple[np.ndarray, np.ndarray]],
        ts_cfg: "TSConfig",
    ) -> np.ndarray:
        """Retrieve a cached or newly computed ideal U_qr vector.

        This routine ensures the required angle data are available and
        returns the ideal U_qr amplitude vector for the specified n and q.

        Args:
            n: Log2 problem size.
            q: U_qr order parameter.
            make_t_s: Callable that generates (t, s) from N and TSConfig.
            ts_cfg: Configuration used for data generation.

        Returns:
            A complex-valued NumPy array of ideal U_qr amplitudes.

        Raises:
            None.
        """
        key = (n, q)
        if key not in self.uqr_ideal_by_nq:
            _, _, _, phi, theta = self.ensure_angle_data(n, make_t_s, ts_cfg)
            self.uqr_ideal_by_nq[key] = u_qr_ideal_from_phi_theta(phi, theta, q)
        return self.uqr_ideal_by_nq[key]

    def get_alpha_prime(self, r: int, K: int) -> np.ndarray:
        """Retrieve cached or newly computed alpha-prime coefficients.

        This routine returns the coefficient vector used in the LCU
        construction for the specified r and K pair.

        Args:
            r: Target approximation order.
            K: Number of LCU terms.

        Returns:
            A NumPy array of alpha-prime coefficients.

        Raises:
            None.
        """
        key = (r, K)
        if key not in self.alpha_prime_by_rK:
            self.alpha_prime_by_rK[key] = alpha_vec_prime(r=r, K=K)
        return self.alpha_prime_by_rK[key]

    def get_ur_ideal(
        self,
        n: int,
        r: int,
        K: int,
        make_t_s: Callable[[int, "TSConfig"], Tuple[np.ndarray, np.ndarray]],
        ts_cfg: "TSConfig",
    ) -> np.ndarray:
        """Retrieve a cached or newly computed ideal U_r vector.

        This routine combines cached ideal U_qr vectors with cached
        alpha-prime coefficients to build and store the ideal U_r
        reference vector.

        Args:
            n: Log2 problem size.
            r: Target approximation order.
            K: Number of LCU terms.
            make_t_s: Callable that generates (t, s) from N and TSConfig.
            ts_cfg: Configuration used for data generation.

        Returns:
            A complex-valued NumPy array of ideal U_r amplitudes.

        Raises:
            None.
        """
        key = (n, r, K)
        if key not in self.ur_ideal_by_nrK:
            coeffs = self.get_alpha_prime(r, K)[:, None]
            uq = np.vstack([
                self.get_uqr_ideal(n, q, make_t_s, ts_cfg)
                for q in range(K)
            ])
            self.ur_ideal_by_nrK[key] = np.sum(coeffs * uq, axis=0)
        return self.ur_ideal_by_nrK[key]


# =============================================================================
# Statevector post-processing
# =============================================================================
def _align_global_phase(a: np.ndarray, b: np.ndarray) -> Tuple[np.ndarray, complex]:
    """Align one statevector to another by removing relative global phase.

    This routine computes the complex phase implied by the inner product
    between two vectors and applies the conjugate phase to the first
    vector.

    Args:
        a: Vector to phase-align.
        b: Reference vector.

    Returns:
        A tuple containing the aligned version of a and the extracted
        phase factor.

    Raises:
        None.
    """
    ip = np.vdot(a, b)
    if abs(ip) == 0:
        return a, 1.0 + 0.0j
    phase = ip / abs(ip)
    return a * np.conj(phase), phase


def extract_uqr_amplitudes(
    sv: Statevector,
    *,
    n: int,
    phi_len: int,
    theta_len: int,
) -> np.ndarray:
    """Extract the computational amplitudes relevant to U_qr verification.

    This routine returns the first 2^n amplitudes from the simulated
    statevector, corresponding to the main data register used for U_qr
    comparison.

    Args:
        sv: Simulated statevector.
        n: Number of qubits in the main data register.
        phi_len: Length of the phi register.
        theta_len: Length of the theta register.

    Returns:
        A complex-valued NumPy array of extracted amplitudes.

    Raises:
        None.
    """
    del phi_len, theta_len
    N = 2**n
    return np.asarray(sv.data[:N], dtype=complex)


def extract_uur_amplitudes(
    sv: Statevector,
    *,
    n: int,
    L: int,
) -> np.ndarray:
    """Extract the computational amplitudes relevant to U_ur LCU verification.

    This routine samples the statevector at a stride determined by the
    ancillary LCU register size so only the target branch amplitudes are
    retained.

    Args:
        sv: Simulated statevector.
        n: Number of qubits in the main data register.
        L: Number of ancilla qubits encoding the LCU branch index.

    Returns:
        A complex-valued NumPy array of extracted amplitudes.

    Raises:
        None.
    """
    N = 2**n
    stride = 1 << L
    return np.asarray(sv.data[0:(N * stride):stride], dtype=complex)


# =============================================================================
# Plot helpers
# =============================================================================
def plot_amplitude_comparison(
    *,
    x: np.ndarray,
    ideal: np.ndarray,
    sim: np.ndarray,
    ylabel: str = "Amplitude",
    save_base: Path | None = None,
    show: bool = False,
) -> None:
    """Plot predicted and simulated amplitudes over the transformed domain.

    This routine creates a publication-style comparison plot for the real
    and imaginary parts of the predicted and simulated amplitudes, and
    optionally saves PNG and EPS outputs.

    Args:
        x: Domain values used on the horizontal axis.
        ideal: Predicted complex amplitudes.
        sim: Simulated complex amplitudes.
        ylabel: Y-axis label.
        save_base: Base path used when saving output figures, without extension.
        show: Whether to display the plot interactively.

    Returns:
        None.

    Raises:
        None.
    """
    _apply_publication_rcparams()

    order = np.argsort(x)
    x_sorted = np.asarray(x)[order]
    ideal_sorted = np.asarray(ideal)[order]
    sim_sorted = np.asarray(sim)[order]

    fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)

    real_color = "tab:blue"
    imag_color = "tab:orange"

    predicted_style = dict(
        linestyle=":",
        marker="o",
        linewidth=1.8,
        markersize=4.5,
    )
    simulated_style = dict(
        linestyle="-",
        marker="o",
        linewidth=1.8,
        markersize=4.5,
    )

    ax.plot(
        x_sorted,
        ideal_sorted.real,
        color=real_color,
        label="Predicted Re",
        **predicted_style,
    )
    ax.plot(
        x_sorted,
        sim_sorted.real,
        color=real_color,
        label="Simulated Re",
        **simulated_style,
    )
    ax.plot(
        x_sorted,
        ideal_sorted.imag,
        color=imag_color,
        label="Predicted Im",
        **predicted_style,
    )
    ax.plot(
        x_sorted,
        sim_sorted.imag,
        color=imag_color,
        label="Simulated Im",
        **simulated_style,
    )

    ax.set_xlabel(r"$x = t - s/N$")
    ax.set_ylabel(ylabel)

    xmin = np.min(x_sorted)
    xmax = np.max(x_sorted)

    xmin_lim = xmin - 0.025
    xmax_lim = xmax + 0.025

    ax.set_xlim(xmin_lim, xmax_lim)

    num_ticks = 5
    ticks = np.round(np.linspace(xmin_lim, xmax_lim, num_ticks), 1)
    ax.set_xticks(ticks)

    leg = ax.legend(
        loc="upper right",
        bbox_to_anchor=(0.98, 0.98),
        fontsize=9,
        handlelength=2.0,
        labelspacing=0.3,
        borderpad=0.3,
        handletextpad=0.4,
        borderaxespad=0.2,
        frameon=True,
    )

    if leg is not None and leg.get_frame() is not None:
        leg.get_frame().set_alpha(1.0)

    _force_opaque(ax)

    if save_base is not None:
        fig.savefig(f"{save_base}.png", dpi=300, bbox_inches="tight", transparent=False)
        fig.savefig(f"{save_base}.eps", format="eps", bbox_inches="tight", transparent=False)

    if show:
        plt.show()

    plt.close(fig)


# =============================================================================
# Single-point tests
# =============================================================================
def test_uqr_once(
    *,
    n: int,
    m: int,
    p: int,
    q: int,
    t: np.ndarray,
    s: np.ndarray,
    ideal_uqr: np.ndarray | None = None,
    do_plot: bool = False,
    save_plot: bool = False,
    plot_tag: str = "",
    loader_mode: str = "dense",
) -> Dict[str, float]:
    """Run one U_qr circuit build, simulation, and verification test.

    This routine constructs the U_qr circuit for one parameter tuple,
    simulates its statevector, compares the extracted amplitudes against
    the analytic ideal reference, and optionally generates a comparison
    plot.

    Args:
        n: Number of data-register qubits.
        m: Precision or auxiliary parameter for the phi construction.
        p: Precision or auxiliary parameter for the theta construction.
        q: U_qr order parameter.
        t: Nonuniform node array.
        s: Nearest-grid index array.
        ideal_uqr: Optional precomputed ideal U_qr vector.
        do_plot: Whether to display the amplitude comparison plot.
        save_plot: Whether to save the amplitude comparison plot.
        plot_tag: Filename tag for saved plots.
        loader_mode: Loader synthesis mode passed to the circuit builder.

    Returns:
        A dictionary containing timings, error metrics, and circuit
        resource statistics.

    Raises:
        None.
    """
    N = 2**n

    j = QuantumRegister(n, "j")
    phi = QuantumRegister(1 + 2 + m, "phi")
    theta = QuantumRegister(2 + p, "theta")
    tgt = QuantumRegister(1, "tgt")
    qc = QuantumCircuit(j, phi, theta, tgt)

    with timer() as t_build:
        U_qr_circuit_no_x(
            n=n,
            m=m,
            p=p,
            q=q,
            t=t,
            s=s,
            uncompute_angles=True,
            loader_mode=loader_mode,
            j=j,
            phi_reg=phi,
            theta_reg=theta,
            tgt=tgt,
            qc=qc,
        )
    build_s = t_build()

    with timer() as t_sim:
        sv = Statevector.from_label("0" * qc.num_qubits).evolve(qc)
    sim_s = t_sim()

    with timer() as t_verify:
        amps = extract_uqr_amplitudes(sv, n=n, phi_len=len(phi), theta_len=len(theta))
        ideal = ideal_uqr if ideal_uqr is not None else u_qr_ideal(t=t, s=s, N=N, q=q)
        amps_raw = np.sqrt(N) * amps
        amps_aligned, _ = _align_global_phase(amps_raw, ideal)
        err = np.linalg.norm(amps_aligned - ideal)

        denom = np.vdot(amps_aligned, amps_aligned).real * np.vdot(ideal, ideal).real
        fid = 0.0 if denom == 0 else abs(np.vdot(amps_aligned, ideal)) ** 2 / denom

    verify_s = t_verify()

    plot_s = 0.0
    if do_plot or save_plot:
        with timer() as t_plot:
            y = t - s / N
            x = N * y
            tag = plot_tag or f"n{n}_m{m}_p{p}_q{q}"
            save_base = FIG_DIR / f"uqr_amp_domain_{tag}" if save_plot else None

            plot_amplitude_comparison(
                x=x,
                ideal=ideal,
                sim=amps_aligned,
                ylabel=r"$u_{q}(x)$",
                save_base=save_base,
                show=do_plot,
            )
        plot_s = t_plot()

    return {
        "build_s": float(build_s),
        "sim_s": float(sim_s),
        "verify_s": float(verify_s),
        "plot_s": float(plot_s),
        "l2_error": float(err),
        "fidelity": float(fid),
        "qubits": float(qc.num_qubits),
        "depth": float(qc.depth()),
        "size": float(qc.size()),
    }


def test_uur_lcu_once(
    *,
    n: int,
    m: int,
    p: int,
    r: int,
    K: int,
    t: np.ndarray,
    s: np.ndarray,
    ideal_ur: np.ndarray | None = None,
    do_plot: bool = False,
    save_plot: bool = False,
    plot_tag: str = "",
    loader_mode: str = "dense",
) -> Dict[str, float]:
    """Run one U_ur LCU circuit build, simulation, and verification test.

    This routine constructs the U_ur LCU circuit for one parameter tuple,
    simulates its statevector, compares the extracted amplitudes against
    the analytic ideal reference, and optionally generates a comparison
    plot.

    Args:
        n: Number of data-register qubits.
        m: Precision or auxiliary parameter for the phi construction.
        p: Precision or auxiliary parameter for the theta construction.
        r: Target approximation order.
        K: Number of LCU terms.
        t: Nonuniform node array.
        s: Nearest-grid index array.
        ideal_ur: Optional precomputed ideal U_r vector.
        do_plot: Whether to display the amplitude comparison plot.
        save_plot: Whether to save the amplitude comparison plot.
        plot_tag: Filename tag for saved plots.
        loader_mode: Loader synthesis mode passed to the circuit builder.

    Returns:
        A dictionary containing alpha normalization, timings, error
        metrics, and circuit resource statistics.

    Raises:
        None.
    """
    N = 2**n
    L = int(np.log2(K))

    with timer() as t_build:
        qc, alpha_ur = U_ur_lcu_circuit(
            n=n,
            m=m,
            p=p,
            r=r,
            K=K,
            t=t,
            s=s,
            uncompute_angles=True,
            loader_mode=loader_mode,
        )
    build_s = t_build()

    with timer() as t_sim:
        sv = Statevector.from_label("0" * qc.num_qubits).evolve(qc)
    sim_s = t_sim()

    with timer() as t_verify:
        amps = extract_uur_amplitudes(sv, n=n, L=L)
        ideal = ideal_ur if ideal_ur is not None else u_r_ideal(t=t, s=s, N=N, r=r, K=K)
        amps_raw = alpha_ur * np.sqrt(N) * amps
        amps_aligned, _ = _align_global_phase(amps_raw, ideal)
        err = np.linalg.norm(amps_aligned - ideal)

        denom = np.vdot(amps_aligned, amps_aligned).real * np.vdot(ideal, ideal).real
        fid = 0.0 if denom == 0 else abs(np.vdot(amps_aligned, ideal)) ** 2 / denom
    verify_s = t_verify()

    plot_s = 0.0
    if do_plot or save_plot:
        with timer() as t_plot:
            y = t - s / N
            x = N * y
            tag = plot_tag or f"n{n}_m{m}_p{p}_r{r}_K{K}"
            save_base = FIG_DIR / f"uur_lcu_amp_domain_{tag}" if save_plot else None

            plot_amplitude_comparison(
                x=x,
                ideal=ideal,
                sim=amps_aligned,
                ylabel=r"$u_r(x)$",
                save_base=save_base,
                show=do_plot,
            )
        plot_s = t_plot()

    return {
        "alpha_ur": float(alpha_ur),
        "build_s": float(build_s),
        "sim_s": float(sim_s),
        "verify_s": float(verify_s),
        "plot_s": float(plot_s),
        "l2_error": float(err),
        "fidelity": float(fid),
        "qubits": float(qc.num_qubits),
        "depth": float(qc.depth()),
        "size": float(qc.size()),
    }


# =============================================================================
# t/s generation
# =============================================================================
@dataclass
class TSConfig:
    """Store configuration parameters for generating t/s test data.

    This configuration object controls the random seed and node jitter
    used when building perturbed-uniform test instances.

    Args:
        None.

    Returns:
        A TSConfig instance containing generation parameters.

    Raises:
        None.
    """

    seed: int = 123
    jitter: float = 0.48


def default_make_t_s(N: int, ts_cfg: "TSConfig | None" = None) -> Tuple[np.ndarray, np.ndarray]:
    """Generate perturbed-uniform nodes and nearest-grid indices.

    This routine builds a reproducible nonuniform node set using the
    supplied configuration and computes the nearest-grid index for each
    node.

    Args:
        N: Problem size.
        ts_cfg: Optional generation configuration.

    Returns:
        A tuple (t, s) of NumPy arrays.

    Raises:
        None.
    """
    if ts_cfg is None:
        ts_cfg = TSConfig()

    seed = getattr(ts_cfg, "seed", None)
    jitter = float(getattr(ts_cfg, "jitter", 0.25))

    rng = np.random.default_rng(seed)
    _ = jitter
    _ = nodes_perturbed_uniform  # keeps import available if you switch generators
    t = nodes_stratified_residuals(N, jitter=0.1, rng=rng)
    s = nearest_grid_indices(t, N)
    return t, s


def build_shared_ts_cache(
    n_values,
    make_t_s: Callable[[int, TSConfig], Tuple[np.ndarray, np.ndarray]] = default_make_t_s,
    ts_cfg: TSConfig = TSConfig(),
) -> Dict[int, Tuple[np.ndarray, np.ndarray]]:
    """Precompute shared (t, s) data for multiple problem sizes.

    This routine generates and stores one node/index pair for each n in
    the provided sweep values so later tests can reuse identical input
    data.

    Args:
        n_values: Iterable of log2 problem sizes.
        make_t_s: Callable that generates (t, s) from N and TSConfig.
        ts_cfg: Configuration used for data generation.

    Returns:
        A dictionary mapping n to (t, s) tuples.

    Raises:
        None.
    """
    ts_by_n: Dict[int, Tuple[np.ndarray, np.ndarray]] = {}
    for n in n_values:
        n_int = int(n)
        N = 2**n_int
        ts_by_n[n_int] = make_t_s(N, ts_cfg)
    return ts_by_n


def make_t_s_from_cache(
    ts_by_n: Dict[int, Tuple[np.ndarray, np.ndarray]]
) -> Callable[[int, TSConfig], Tuple[np.ndarray, np.ndarray]]:
    """Create a t/s generator that reads from a precomputed cache.

    This routine returns a callable compatible with the standard make_t_s
    interface but serving values from an existing dictionary keyed by n.

    Args:
        ts_by_n: Dictionary mapping n to cached (t, s) tuples.

    Returns:
        A callable that retrieves cached (t, s) values for a given N.

    Raises:
        None.
    """
    def _make_t_s(N: int, ts_cfg: TSConfig) -> Tuple[np.ndarray, np.ndarray]:
        """Retrieve cached (t, s) data for a power-of-two size N.

        This nested routine converts N to n = log2(N), validates that N is
        a power of two, and returns the matching cached data.

        Args:
            N: Problem size.
            ts_cfg: Unused compatibility argument.

        Returns:
            A tuple (t, s) of cached NumPy arrays.

        Raises:
            ValueError: If N is not a power of two.
            KeyError: If no cached data are available for the inferred n.
        """
        del ts_cfg
        n = int(np.log2(N))
        if 2**n != N:
            raise ValueError(f"N={N} is not a power of two.")
        if n not in ts_by_n:
            raise KeyError(f"No cached (t, s) available for n={n}.")
        return ts_by_n[n]

    return _make_t_s


# =============================================================================
# Sweep runners
# =============================================================================
def run_tests_uqr(
    *,
    n_values,
    m_values,
    p_values,
    q_values,
    make_t_s: Callable[[int, TSConfig], Tuple[np.ndarray, np.ndarray]] = default_make_t_s,
    ts_cfg: TSConfig = TSConfig(),
    cache: SweepCache | None = None,
    warm_cache: bool = False,
    do_plot: bool = False,
    save_plot: bool = False,
    save_outputs: bool = True,
    top_k: int = 20,
    loader_mode: str = "dense",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run a full parameter sweep for U_qr tests.

    This routine iterates over all combinations of n, m, p, and q,
    reuses cached analytic data when available, executes one test per
    point, prints progress lines, and returns complete and worst-case
    result tables.

    Args:
        n_values: Iterable of data-register sizes in log2 form.
        m_values: Iterable of phi-related precision parameters.
        p_values: Iterable of theta-related precision parameters.
        q_values: Iterable of U_qr order parameters.
        make_t_s: Callable that generates (t, s) from N and TSConfig.
        ts_cfg: Configuration used for data generation.
        cache: Optional shared analytic cache.
        warm_cache: Whether to precompute angle data before the sweep.
        do_plot: Whether to display per-case plots.
        save_plot: Whether to save per-case plots.
        save_outputs: Whether to save CSV and JSON result files.
        top_k: Number of worst cases to retain.
        loader_mode: Loader synthesis mode passed through to each test case.

    Returns:
        A tuple (df, worst) of the full results DataFrame and the
        worst-case DataFrame.

    Raises:
        None.
    """
    if save_plot or save_outputs:
        _ensure_dirs()

    rows: List[dict] = []
    total_t0 = time.perf_counter()
    cache = cache or SweepCache()

    if warm_cache:
        for n in n_values:
            cache.ensure_angle_data(int(n), make_t_s, ts_cfg)

    print_section("U_qr sweep")

    for idx, (n, m, p, q_) in enumerate(itertools.product(n_values, m_values, p_values, q_values), 1):
        n = int(n)
        m = int(m)
        p = int(p)
        q_ = int(q_)

        t_vec, s_vec = cache.get_ts(n, make_t_s, ts_cfg)
        ideal_uqr = cache.get_uqr_ideal(n, q_, make_t_s, ts_cfg)

        t0_point = time.perf_counter()
        res = test_uqr_once(
            n=n,
            m=m,
            p=p,
            q=q_,
            t=t_vec,
            s=s_vec,
            ideal_uqr=ideal_uqr,
            do_plot=do_plot,
            save_plot=save_plot,
            plot_tag=f"case{idx:03d}_n{n}_m{m}_p{p}_q{q_}",
            loader_mode=loader_mode,
        )
        point_s = time.perf_counter() - t0_point

        row = dict(
            case_idx=idx,
            n=n,
            m=m,
            p=p,
            q=q_,
            build_s=float(res["build_s"]),
            sim_s=float(res["sim_s"]),
            verify_s=float(res["verify_s"]),
            plot_s=float(res["plot_s"]),
            total_point_s=float(point_s),
            l2_error=float(res["l2_error"]),
            fidelity=float(res["fidelity"]),
            qubits=int(res["qubits"]),
            depth=int(res["depth"]),
            size=int(res["size"]),
        )
        rows.append(row)

        print(
            f"[{idx:03d}] "
            f"(n={n}, m={m}, p={p}, q={q_})  "
            f"build={row['build_s']:.3f}s  "
            f"sim={row['sim_s']:.3f}s  "
            f"verify={row['verify_s']:.3f}s  "
            f"total={row['total_point_s']:.3f}s  "
            f"err={row['l2_error']:.3e}  "
            f"fid={row['fidelity']:.10f}"
        )

    df = pd.DataFrame(rows)
    worst = extract_worst_cases(df, metric="l2_error", top_k=top_k)

    if save_outputs:
        tag = f"uqr_{int(time.time())}"
        df.to_csv(FIG_DIR / f"all_results_{tag}.csv", index=False)
        worst.to_json(FIG_DIR / f"worst_cases_top{top_k}_{tag}.json", orient="records", indent=2)

    total_s = time.perf_counter() - total_t0
    print()
    print(f"(U_qr) TOTAL SWEEP RUNTIME: {total_s:.3f} seconds")

    return df, worst


def run_tests_uur_lcu(
    *,
    n_values,
    m_values,
    p_values,
    r_values,
    K_values,
    make_t_s: Callable[[int, TSConfig], Tuple[np.ndarray, np.ndarray]] = default_make_t_s,
    ts_cfg: TSConfig = TSConfig(),
    cache: SweepCache | None = None,
    warm_cache: bool = False,
    do_plot: bool = False,
    save_plot: bool = False,
    save_outputs: bool = True,
    top_k: int = 20,
    loader_mode: str = "dense",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run a full parameter sweep for U_ur LCU tests.

    This routine iterates over all combinations of n, m, p, r, and K,
    reuses cached analytic data when available, executes one test per
    point, prints progress lines, and returns complete and worst-case
    result tables.

    Args:
        n_values: Iterable of data-register sizes in log2 form.
        m_values: Iterable of phi-related precision parameters.
        p_values: Iterable of theta-related precision parameters.
        r_values: Iterable of target approximation orders.
        K_values: Iterable of LCU term counts.
        make_t_s: Callable that generates (t, s) from N and TSConfig.
        ts_cfg: Configuration used for data generation.
        cache: Optional shared analytic cache.
        warm_cache: Whether to precompute angle data before the sweep.
        do_plot: Whether to display per-case plots.
        save_plot: Whether to save per-case plots.
        save_outputs: Whether to save CSV and JSON result files.
        top_k: Number of worst cases to retain.
        loader_mode: Loader synthesis mode passed through to each test case.

    Returns:
        A tuple (df, worst) of the full results DataFrame and the
        worst-case DataFrame.

    Raises:
        ValueError: If any K value is not a positive power of two.
    """
    if save_plot or save_outputs:
        _ensure_dirs()

    rows: List[dict] = []
    total_t0 = time.perf_counter()
    cache = cache or SweepCache()

    if warm_cache:
        for n in n_values:
            cache.ensure_angle_data(int(n), make_t_s, ts_cfg)

    print_section("U_ur (LCU) sweep")

    for idx, (n, m, p, r, K) in enumerate(
        itertools.product(n_values, m_values, p_values, r_values, K_values), 1
    ):
        n = int(n)
        m = int(m)
        p = int(p)
        r = int(r)
        K = int(K)

        if K <= 0 or (K & (K - 1)) != 0:
            raise ValueError(f"K must be a power of two; got K={K}.")

        t_vec, s_vec = cache.get_ts(n, make_t_s, ts_cfg)
        ideal_ur = cache.get_ur_ideal(n, r, K, make_t_s, ts_cfg)

        t0_point = time.perf_counter()
        res = test_uur_lcu_once(
            n=n,
            m=m,
            p=p,
            r=r,
            K=K,
            t=t_vec,
            s=s_vec,
            ideal_ur=ideal_ur,
            do_plot=do_plot,
            save_plot=save_plot,
            plot_tag=f"case{idx:03d}_n{n}_m{m}_p{p}_r{r}_K{K}",
            loader_mode=loader_mode,
        )
        point_s = time.perf_counter() - t0_point

        row = dict(
            case_idx=idx,
            n=n,
            m=m,
            p=p,
            r=r,
            K=K,
            alpha_ur=float(res["alpha_ur"]),
            build_s=float(res["build_s"]),
            sim_s=float(res["sim_s"]),
            verify_s=float(res["verify_s"]),
            plot_s=float(res["plot_s"]),
            total_point_s=float(point_s),
            l2_error=float(res["l2_error"]),
            fidelity=float(res["fidelity"]),
            qubits=int(res["qubits"]),
            depth=int(res["depth"]),
            size=int(res["size"]),
        )
        rows.append(row)

        print(
            f"[{idx:03d}] "
            f"(n={n}, m={m}, p={p}, r={r}, K={K})  "
            f"build={row['build_s']:.3f}s  "
            f"sim={row['sim_s']:.3f}s  "
            f"verify={row['verify_s']:.3f}s  "
            f"total={row['total_point_s']:.3f}s  "
            f"alpha={row['alpha_ur']:.6g}  "
            f"err={row['l2_error']:.3e}  "
            f"fid={row['fidelity']:.10f}"
        )

    df = pd.DataFrame(rows)
    worst = extract_worst_cases(df, metric="l2_error", top_k=top_k)

    if save_outputs:
        tag = f"uur_lcu_{int(time.time())}"
        df.to_csv(FIG_DIR / f"all_results_{tag}.csv", index=False)
        worst.to_json(FIG_DIR / f"worst_cases_top{top_k}_{tag}.json", orient="records", indent=2)

    total_s = time.perf_counter() - total_t0
    print()
    print(f"(U_ur LCU) TOTAL SWEEP RUNTIME: {total_s:.3f} seconds")

    return df, worst


# =============================================================================
# Report builders
# =============================================================================
def report_uqr_results(
    df_uqr: pd.DataFrame,
    worst_uqr: pd.DataFrame,
) -> None:
    """Print the summary report for U_qr sweep results.

    This routine prints a worst-case error table for the completed U_qr
    sweep.

    Args:
        df_uqr: Full U_qr results DataFrame.
        worst_uqr: Worst-case U_qr rows.

    Returns:
        None.

    Raises:
        None.
    """
    print_subsection("Worst cases by l2_error")
    print_worst_cases_table_uqr(worst_uqr)


def report_uur_results(
    df_uur: pd.DataFrame,
    worst_uur: pd.DataFrame,
) -> None:
    """Print the summary report for U_ur LCU sweep results.

    This routine prints an aggregate sweep summary and a worst-case error
    table for the completed U_ur LCU sweep.

    Args:
        df_uur: Full U_ur LCU results DataFrame.
        worst_uur: Worst-case U_ur LCU rows.

    Returns:
        None.

    Raises:
        None.
    """
    print_section("U_ur (LCU) results summary")
    print_sweep_summary(df_uur, label="U_ur(LCU)")

    print_subsection("Worst cases by l2_error")
    print_worst_cases_table_uur(worst_uur)

Repo root: /Users/junaida/Documents/Non-Uniform-QFT
Added to sys.path: /Users/junaida/Documents/Non-Uniform-QFT/src


In [2]:
# =============================================================================
# Main
# =============================================================================
if __name__ == "__main__":
    # -------------------------------------------------------------------------
    # Select loader mode for this sweep block.
    # Use "dense" for dense-unitary loaders or "mcx" for MCX decomposition.
    # -------------------------------------------------------------------------
    loader_mode = "dense"

    total_start = time.perf_counter()

    n_values = range(3, 7)
    m_values = range(4, 5)
    p_values = range(5, 6)
    q_values = range(0, 3)

    ts_cfg = TSConfig(seed=123, jitter=0.1)

    ts_by_n = build_shared_ts_cache(
        n_values=n_values,
        make_t_s=default_make_t_s,
        ts_cfg=ts_cfg,
    )
    shared_make_t_s = make_t_s_from_cache(ts_by_n)

    cache = SweepCache(ts_by_n=dict(ts_by_n))

    # -------------------------------------------------------------------------
    # U_qr sweep
    # -------------------------------------------------------------------------
    df_uqr, worst_uqr = run_tests_uqr(
        n_values=n_values,
        m_values=m_values,
        p_values=p_values,
        q_values=q_values,
        make_t_s=shared_make_t_s,
        ts_cfg=ts_cfg,
        cache=cache,
        warm_cache=True,
        do_plot=False,
        save_plot=True,
        save_outputs=False,
        top_k=20,
        loader_mode=loader_mode,
    )

    report_uqr_results(
        df_uqr,
        worst_uqr,
    )


U_qr sweep
[001] (n=3, m=4, p=5, q=0)  build=0.111s  sim=0.208s  verify=0.000s  total=0.926s  err=1.437e-01  fid=0.9975036152
[002] (n=3, m=4, p=5, q=1)  build=0.088s  sim=0.260s  verify=0.000s  total=0.766s  err=9.789e-02  fid=0.9963890334
[003] (n=3, m=4, p=5, q=2)  build=0.021s  sim=0.270s  verify=0.000s  total=0.709s  err=1.081e-01  fid=0.9970616774
[004] (n=4, m=4, p=5, q=0)  build=0.503s  sim=0.572s  verify=0.000s  total=1.510s  err=1.495e-01  fid=0.9986725267
[005] (n=4, m=4, p=5, q=1)  build=0.518s  sim=0.815s  verify=0.000s  total=1.787s  err=9.268e-02  fid=0.9984112206
[006] (n=4, m=4, p=5, q=2)  build=0.066s  sim=0.813s  verify=0.000s  total=1.364s  err=1.750e-01  fid=0.9959813032
[007] (n=5, m=4, p=5, q=0)  build=3.709s  sim=1.933s  verify=0.000s  total=6.244s  err=2.050e-01  fid=0.9986869125
[008] (n=5, m=4, p=5, q=1)  build=3.788s  sim=3.682s  verify=0.000s  total=8.345s  err=1.281e-01  fid=0.9984663793
[009] (n=5, m=4, p=5, q=2)  build=0.694s  sim=4.174s  verify=0.000s 

In [3]:
# -------------------------------------------------------------------------
# U_r (LCU) sweep
# -------------------------------------------------------------------------
if __name__ == "__main__":
    loader_mode = "mcx"

    total_start = time.perf_counter()

    n_values = range(3, 6)
    r_values = range(0, 3)
    K_values = [2]

    df_uur, worst_uur = run_tests_uur_lcu(
        n_values=n_values,
        m_values=m_values,
        p_values=p_values,
        r_values=r_values,
        K_values=K_values,
        make_t_s=shared_make_t_s,
        ts_cfg=ts_cfg,
        cache=cache,
        warm_cache=True,
        do_plot=False,
        save_plot=True,
        save_outputs=False,
        top_k=20,
        loader_mode=loader_mode,
    )

    report_uur_results(
        df_uur,
        worst_uur,
    )


U_ur (LCU) sweep
[001] (n=3, m=4, p=5, r=0, K=2)  build=0.067s  sim=18.535s  verify=0.000s  total=19.007s  alpha=0.725277  err=1.042e-01  fid=0.9975036152
[002] (n=3, m=4, p=5, r=1, K=2)  build=0.089s  sim=32.751s  verify=0.000s  total=33.241s  alpha=1.23721  err=1.211e-01  fid=0.9963890334
[003] (n=3, m=4, p=5, r=2, K=2)  build=0.045s  sim=21.458s  verify=0.000s  total=21.974s  alpha=0.263811  err=3.790e-02  fid=0.9975036152
[004] (n=4, m=4, p=5, r=0, K=2)  build=0.532s  sim=378.664s  verify=0.000s  total=379.602s  alpha=0.725277  err=1.084e-01  fid=0.9986725267
[005] (n=4, m=4, p=5, r=1, K=2)  build=0.521s  sim=649.940s  verify=0.000s  total=650.864s  alpha=1.23721  err=1.147e-01  fid=0.9984112206
[006] (n=4, m=4, p=5, r=2, K=2)  build=0.263s  sim=325.365s  verify=0.000s  total=326.008s  alpha=0.263811  err=3.944e-02  fid=0.9986725267
[007] (n=5, m=4, p=5, r=0, K=2)  build=1.121s  sim=1987.640s  verify=0.000s  total=1989.155s  alpha=0.725277  err=1.487e-01  fid=0.9986869125
[008] (n